# FastAPI + Docker Fundamentals

Every agent this curriculum has built lives inside a single Python process — a notebook kernel (001-008) or, in 009, a Gradio server you started by hand and closed when you were done. This notebook is about the two tools that turn "runs on my machine while I'm watching it" into "runs as a real, independently-callable service, packaged so it runs the same way anywhere."

**Why this exists as its own module, before the Capstone:** this curriculum has consistently introduced a new tool only once there's a concrete problem motivating it — but that problem doesn't have to be the *biggest* one. FastAPI and Docker don't need the full, integrated Capstone system to exist before they're worth learning; they just need *something* to practice on. So this notebook deliberately stays toy-scale — wrapping one already-built agent from 008 — so that by the time 012 (Capstone) asks you to do the real version (the complete system, real secrets, an actual deploy), the tools themselves are already familiar and only the scale is new.

## What this notebook covers
1. What an API is (reused, not re-taught), what FastAPI specifically adds, and the simplest possible app
2. Real request/response models — the same Pydantic mechanism you already know
3. Wiring in an actual agent from 008, not a new toy example
4. Async endpoints — and a real, measured reason they matter
5. The auto-generated docs you get for free
6. What Docker actually is, and getting it installed
7. A real Dockerfile, built and run
8. Secrets in a container — the payoff of a promise 008 made

Closing: what's still not real deployment

## Setup

In [5]:
# %pip install -U "fastapi[standard]" langchain langchain-anthropic langgraph
# fastapi[standard] (not just fastapi) is what installs the `fastapi` CLI command
# used throughout this notebook -- plain `pip install fastapi` alone leaves you
# with the library but not the command-line tool.

import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent

def get_llm(max_tokens=256, temperature=0.2):
    return ChatAnthropic(
        model="claude-haiku-4-5",
        max_tokens=max_tokens,
        temperature=temperature,
        anthropic_api_key=ANTHROPIC_API_KEY,
    )

print("Setup ready.")

Setup ready.


**⚠️ Needs your API key** for anything that actually calls the model — but a real amount of this notebook doesn't need one at all. FastAPI's own mechanics (routing, validation, the sync/async difference) are genuine Python/web behavior that has nothing to do with Claude, and every one of those cells was actually run, with real servers and real `curl` requests, while building this. Only the cells that wire in a real agent need your key.

**A different execution convention from 008/009, worth naming:** this notebook runs real local web servers in the background, tests them with `curl`, then shuts them down — inside a single cell rather than split across cells the way 001-009 have done, since a server needs to be running *and* stopped within the same logical unit of work for the output to make sense on its own.

## 1. What an API is (already known), and what FastAPI specifically adds

### Not re-taught — `ai_engineer_tutorial.ipynb` already covered this

Section 9 of that very first theory notebook gave you the restaurant analogy: *"You're the customer, the kitchen is some service with data or functionality you want, and the waiter is the API. You don't go into the kitchen yourself — you tell the waiter what you want, the waiter goes to the kitchen, and comes back with the result."* Every single `client.messages.create(...)` call, every `llm.invoke(...)`, this entire curriculum — all of it has been you, the customer, talking to Anthropic's API, the waiter, without ever touching Claude's kitchen directly.

What's new here isn't the concept — it's which side of the counter you're standing on. Every API this curriculum has *called* so far (Anthropic's, LangSmith's) was built and run by someone else. This notebook is about **being the restaurant** — building the waiter yourself, so someone else's code can be the customer calling *your* agent, the same way your code has been calling Claude's.

### What FastAPI specifically is

A Python library for building exactly that: you write plain Python functions, decorate them to say "this one handles requests to this URL," and FastAPI turns that into a real, running web server that speaks HTTP — the actual protocol browsers, `curl`, and other programs use to talk to any web service, including Anthropic's own API underneath the SDK.

### What that sentence actually means, ground-up

**The core problem: two separate programs, possibly on two different physical computers, need to talk to each other.** They can't just call each other's Python functions directly -- they might not be in the same language, the same machine, or even the same building. So they send actual messages back and forth over a network, the same way two people communicate by mail even without speaking face to face.

**A protocol** is just an agreed-upon set of rules for how those messages have to be formatted, so both sides understand each other -- not a technical mystery, the same idea as a phone call needing both people to agree "I'll say hello, then you say hello back." Without an agreed format, one program's message just looks like garbage to the other.

**HTTP** (HyperText Transfer Protocol) is *the* specific protocol almost the entire web runs on -- the exact rules for "how do I ask another program for something, and how does it hand back an answer." It defines things like: a message has a *type* (`GET` -- "give me something," `POST` -- "here's some data, do something with it," the same `POST` used in every `curl -X POST` in this notebook), a *destination*, and a *body* (the actual content, if any).

**A URL** is that destination -- an address saying exactly *where* to send the message and *what* you're asking for. `http://127.0.0.1:8000/chat` breaks down as: `http://` (use HTTP), `127.0.0.1:8000` (which machine, and which "door" on it -- the port -- to knock on), `/chat` (which specific thing on that machine, matching the `@app.post("/chat")` decorator from Section 2).

**A request** is one message, sent following HTTP's rules, to a URL -- literally what every `curl -X POST http://127.0.0.1:8000/chat -d '{"message": "hello"}'` in this notebook has been doing: build an HTTP message, send it to that address.

**A web server** is a program that sits there, running continuously, listening for HTTP requests, and sending back an HTTP **response** for each one. That's exactly what `fastapi run` starts -- before running it, nothing was listening at `127.0.0.1:8000`; after, there's a real program waiting there, and FastAPI's whole job (the "under the hood" cell below) is turning each incoming HTTP request into an ordinary Python function call, and turning that function's return value back into an HTTP response.

**A browser** (Chrome, Safari) is just a program built to do two things: construct an HTTP request when you type a URL or click a link, and take the HTTP response that comes back and *render* it -- turn raw data into the visual page you see.

**`curl`** does the exact same first half -- build an HTTP request, send it, show you the response -- but skips the rendering entirely, just printing the raw response text to a terminal. That's exactly why it's the right tool for testing an API directly: you don't want a pretty page, you want to see the actual raw data your endpoint sent back.

**The connect-back, since this ties directly into something already known:** every `client.messages.create(...)` / `llm.invoke(...)` call this whole curriculum has been -- underneath the Anthropic SDK, underneath `ChatAnthropic` -- is *also* just an HTTP request, sent to a URL on Anthropic's servers, following HTTP's rules, getting an HTTP response back. The SDK's whole job is building that request and parsing that response so it never has to be thought about directly. This has been happening the entire time; this notebook is just the first place the *server* side of that same exchange gets built, instead of only ever sending requests.

### Why FastAPI specifically, not Flask or Django

Two concrete reasons, not just popularity:
1. **The Pydantic tie-in is real, not incidental.** FastAPI validates every request and response using the exact same `BaseModel` mechanism already used for `Joke`, `JudgeScore`, `SafetyCheck` — Section 3 below makes this literal, not just an analogy.
2. **It's async-native**, which matters concretely for anything calling an LLM: that call is I/O-bound (waiting on a network response, not burning CPU), and Section 5 below actually measures why that distinction changes how many requests your service can handle at once.

### The simplest version that actually works

In [4]:
# # copy this into the 010_app folder as app.py
# from fastapi import FastAPI

# app = FastAPI()

# @app.get("/")
# def read_root():
#     return {"message": "Hello from FastAPI"}

**Run this from a terminal, not this notebook** — a FastAPI app is a long-running server, not a single cell's worth of work; save the cell above as `app.py` and run:

```bash
fastapi run Phase\ 2/03_Anthropic_Notes/010_app/app.py
```

(`fastapi run` needs `fastapi[standard]`, not just `fastapi` — the plain package gives you the library, `[standard]` adds the actual `fastapi` command.)

**📌 Note for every "run this yourself" cell in this notebook, not just this one:** two things have to be true first, every time.

1. **Your venv needs to be activated, in that same terminal, first.** `fastapi[standard]` (and `langchain`, `langgraph`, everything else) were installed specifically into the `upskill2` venv this whole curriculum uses — outside it, your terminal's Python doesn't know any of this exists, and the `fastapi` command likely won't even be found.
   ```bash
   source /Users/Sarah/venvs/upskill2/bin/activate
   ```
   You'll know it worked if your prompt shows `(upskill2)` at the start of the line.
2. **You need to actually be inside the folder containing the `.py` file you're running.** This notebook never has you save these cells as real files automatically — copy the code into a real file yourself first (any editor works), in a dedicated folder, then `cd` there before running anything. Suggested layout, reused for every `app.py`/`app2.py`/`app4_2.py`/`app_async.py` in this notebook -- just swap the filename in the last line each time:
   ```bash
   mkdir -p "/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/2026_Upskill/Phase 2/03_Anthropic_Notes/010_app"
   source /Users/Sarah/venvs/upskill2/bin/activate
   cd "/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/2026_Upskill/Phase 2/03_Anthropic_Notes/010_app"
   fastapi run app.py
   ```

Verified for real while building this notebook — here's the exact, genuine output of doing precisely that, then hitting it with `curl` from a second terminal:

```
$ fastapi run app.py

 ⚡️ Starting FastAPI in production mode

 🐍 Using import string: app:app

 🌐 Server started at http://0.0.0.0:8000
    Documentation at http://0.0.0.0:8000/docs

  Logs:

INFO:     Started server process [99983]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)

# --- from a second terminal ---
$ curl http://127.0.0.1:8000/
{"message":"Hello from FastAPI"}
```

### Two terminals, talking to each other — mapped back onto the restaurant analogy

Worth pausing on what just actually happened between those two terminals, since it's the exact same shape as every request this whole curriculum has ever made, just visible for the first time instead of hidden inside an SDK.

**Terminal 1 (running `fastapi run app.py`) is the whole restaurant** — not just the waiter. It's the building, the kitchen, and the waiter all together: FastAPI itself is the waiter part (it receives the order, figures out which "dish" -- which endpoint -- was asked for, and carries the answer back), and your actual Python function (`read_root`, later `chat`) is the kitchen part -- where the real work happens. Building `app.py` and running it *is* building the restaurant; before `fastapi run` started it, there was no restaurant there at all, just a recipe (the code) sitting unused.

**Terminal 2 (running `curl`) is the customer** -- walking up and placing an order (`curl http://127.0.0.1:8000/`), then waiting right there for the food (the response) to come back. Nothing about *being* the customer requires being inside the restaurant, or even on the same street -- which is exactly why terminal 2 doesn't care what folder it's run from, and doesn't need the venv active either (covered a couple messages back): ordering food doesn't require knowing how the kitchen is organized internally, or speaking its "language" (Python) at all -- `curl` only needs to know the restaurant's address and how to place an HTTP order, nothing about what's happening inside.

Terminal 1 *does* care about its folder and venv, for the opposite reason: it's not placing an order, it's *being* the kitchen -- it has to actually find the recipe (`app.py`) and have every ingredient (`fastapi`, `langchain`, etc., from the venv) on hand to cook anything at all.

### 🔍 Under the hood — what actually happened there, mapped onto the restaurant analogy

Recall the setup from the previous section: terminal 1, running `fastapi run app.py`, is the whole restaurant. FastAPI itself plays the **waiter**; your function (`read_root`) is the **kitchen**. Here's the same sequence of events again, this time with every step labeled by which role is doing it.

**Before anyone orders anything -- getting the restaurant ready:**
- `app = FastAPI()` -- this hires the waiter, but the restaurant isn't open yet. Nobody's listening at the door. Same "construction vs. calling it" distinction as `get_llm()` back in 001 -- building the object isn't the same as using it.
- `@app.get("/")` -- this is you handing the waiter a note: *"if anyone orders the dish called `/`, the kitchen recipe for it is `read_root`."* FastAPI keeps every one of these notes in one place internally -- literally just a dictionary, `{"/": read_root}` -- its **routing table**. Nothing gets cooked yet; you're only building the menu.
- `fastapi run app.py` -- the moment the restaurant actually opens. FastAPI hands the front door to `uvicorn` (a doorperson whose only job is physically answering knocks -- accepting network connections), and everyone just waits.

**When `curl` sends the order -- a `GET /` request:**
1. The order arrives at the door (`uvicorn` accepts the network connection) and gets handed to the waiter (FastAPI).
2. The waiter checks the menu note: *"`/`? That's `read_root`'s dish."* -- a routing-table lookup, nothing more.
3. The waiter carries the order back to the kitchen, and the kitchen just cooks, the ordinary way. `read_root()` here is a completely normal Python function call -- no network, no HTTP, nothing unusual from the kitchen's point of view. It has no idea a "customer" or a "waiter" exists; it just does its one job and hands back a plain dict, `{"message": "Hello from FastAPI"}`, the same way any Python function returns anything.
4. The waiter takes that finished dish and **packages it to go** -- this is the "serialized to JSON" step, converting the plain Python dict into the standard format every HTTP response travels in.
5. The waiter carries the packaged order back out to the customer (`curl`, terminal 2), over the network.

**The one sentence worth sitting with:** "Your function never touches HTTP, sockets, or JSON serialization directly" means the kitchen never has to know it's part of a restaurant at all -- `read_root` would work identically as a plain function called from anywhere in Python, with no web server in sight. The waiter (FastAPI) is the only part of this that knows or cares about the network side; that separation -- "the kitchen just cooks, the waiter handles everyone outside the kitchen" -- is the actual design FastAPI is built around, not an incidental detail.

## 2. Real request/response models — the exact `BaseModel` mechanism, applied to HTTP

Section 1's endpoint took no input and always returned the same thing — not useful yet. A real endpoint needs to describe *what it expects* and *what it promises to return*, the same way `with_structured_output(Joke)` back in 001 described a shape for Claude's output. FastAPI uses the identical tool: a `BaseModel` subclass as a type hint on your function's parameter.

**In restaurant terms:** so far the "menu" only had one dish with no options. A real order needs an actual order slip — a fixed form the customer fills in (`ChatRequest`) — and the kitchen needs to promise it'll always plate the finished dish the same standard way (`ChatResponse`), so whoever's waiting for it knows exactly what to expect back.

In [ ]:
# # app2.py
# from fastapi import FastAPI
# from pydantic import BaseModel, Field

# app = FastAPI()

# class ChatRequest(BaseModel):
#     message: str
#     thread_id: str = Field(default="default")

# class ChatResponse(BaseModel):
#     reply: str

# @app.post("/chat", response_model=ChatResponse)
# def chat(request: ChatRequest):
#     # Placeholder logic for now -- the real agent gets wired in next section.
#     return ChatResponse(reply=f"You said: {request.message} (thread={request.thread_id})")

Run it (`fastapi run app2.py`), then a valid request and a deliberately broken one — both genuinely run:

**Exact terminal steps** — same two-terminal shape as Section 1 (terminal 1 = restaurant, terminal 2 = customer), just a new file:

```bash
# Terminal 1 -- the restaurant. Save the code above as app2.py in the same folder as before.
source /Users/Sarah/venvs/upskill2/bin/activate
cd "/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/2026_Upskill/Phase 2/03_Anthropic_Notes/010_app"
fastapi run app2.py
```

Leave that running. Then, in a **second terminal** (no venv or specific folder needed — same "the customer doesn't need to know how the kitchen works" reasoning from Section 1):

```bash
# Terminal 2 -- the customer. A valid order:
curl -X POST http://127.0.0.1:8000/chat \
    -H "Content-Type: application/json" -d '{"message": "hello there"}'

# A deliberately broken order -- missing the required "message" field:
curl -X POST http://127.0.0.1:8000/chat \
    -H "Content-Type: application/json" -d '{}'
```

The first should return `{"reply":"You said: hello there (thread=default)"}`; the second should come back with a `422` and the Pydantic validation error shown a couple cells below — genuinely worth running both, not just the passing one, since the whole point of this section is watching FastAPI reject bad input *before* your function ever sees it.

When you're done, go back to terminal 1 and press `Ctrl+C` to stop the server before moving on to Section 3's `app4_2.py` — otherwise port 8000 will still be occupied by `app2.py` when you try to start the next one.

```
$ curl -X POST http://127.0.0.1:8000/chat \
    -H "Content-Type: application/json" -d '{"message": "hello there"}'
{"reply":"You said: hello there (thread=default)"}

$ curl -X POST http://127.0.0.1:8000/chat \
    -H "Content-Type: application/json" -d '{}'
{"detail":[{"type":"missing","loc":["body","message"],"msg":"Field required","input":{}}]}
```

### 🔍 Under the hood — the second call never reached your function at all

**In restaurant terms first:** the waiter (FastAPI) checks the order slip *before* ever walking it to the kitchen. A slip missing a required field (no `message` written down) gets handed straight back to the customer with "this order's incomplete" — the kitchen (your `chat` function) never even hears that order existed. That's exactly what happened above.

Mechanically: notice `chat(request: ChatRequest)` never ran an `if` statement checking whether `message` was present — there's no validation code anywhere in that function. That's the point: `request: ChatRequest` as a type hint tells FastAPI to construct a real `ChatRequest` object from the incoming JSON *before* calling `chat` at all. When the JSON is missing a required field, `ChatRequest(...)` raises the exact same `ValidationError` you've seen from `Joke`/`JudgeScore`/`SafetyCheck` all session — FastAPI catches that error automatically and turns it into the `422 Unprocessable Entity` response above, without your function's body ever executing. `{"type":"missing","loc":["body","message"],"msg":"Field required"}` isn't a custom error message anyone wrote — it's Pydantic's own validation error, exactly the shape you'd get calling `ChatRequest()` directly in Python, just repackaged as JSON.

`response_model=ChatResponse` does the same validation in the other direction — the kitchen's own promise, checked. If `chat`'s return value didn't actually match `ChatResponse`'s shape, FastAPI would catch that on the way back out, before a malformed plate ever reached the customer. Same mechanism, same class, both directions -- the waiter checks the order going in, and checks the dish going out, against the same two standard forms.

### 🔗 Ties back to theory

This is the concrete answer to something worth naming explicitly: `BaseModel` was introduced in 001 purely to shape *Claude's* output. It turns out to be the same tool for shaping *your own API's* input and output — nothing about `BaseModel` itself is Claude-specific; it's a general "describe and validate the shape of data" tool, and FastAPI is simply the second real place this curriculum has needed exactly that.

### 💬 Question — why did the `curl` commands need `-H "Content-Type: application/json"` at all? That's not part of the schema.

Right that it's not part of `ChatRequest` — it operates one level *before* the schema ever gets involved, and it's worth a concrete picture rather than an abstract rule.

**Picture a sealed envelope arriving in the mail.** On the *outside* there's a label saying what language the letter inside is written in. Inside is the actual letter. You read the label first, and that tells you *how* to read what's inside — if the label says French, you read it as French. If someone mislabels the envelope (writes "French" on it, but the letter's actually in English), you'd try to read English words as French and get nonsense — not because the letter was bad, but because you were told to interpret it the wrong way.

**Tested directly, not just asserted** — sending the exact same, perfectly valid JSON body, once with the header and once without:

```
=== WITH Content-Type: application/json ===
{"reply":"You said: hello there (thread=default)"}

=== WITHOUT any Content-Type header ===
> Content-Type: application/x-www-form-urlencoded   <- curl's own default
{"detail":[{"type":"model_attributes_type","loc":["body"],"msg":"Input should be a valid dictionary or object to extract fields from"...}]}
```

Same body both times. The only difference is the label on the envelope. Without `-H`, `curl -d` defaults to labeling the envelope `application/x-www-form-urlencoded` -- old-school web-form data (`message=hello+there`, not JSON at all) -- because that used to be the standard way browsers submitted forms, long before JSON was common. FastAPI read that label, believed it, and tried to interpret perfectly good JSON as if it were form data -- got confused, and gave up with "not a valid dictionary," never even reaching the point of checking whether a `message` field existed.

`-H "Content-Type: application/json"` is correcting the label to match what's actually inside the envelope.

**Where the schema fits in:** it never gets involved in this specific failure at all. `ChatRequest` is the checklist applied *after* the letter has been successfully read in the right language -- "does it mention a `message`? is it a string?" If the letter couldn't be read at all because the wrong language was assumed, there's no letter yet to run the checklist against -- which is exactly why this error was about failing to extract a dictionary, not about a missing field.

## 3. Wiring in an actual agent — 008's, not a new toy example

Per the "reuse, don't just cite" habit that's helped remembering material so far: the agent this endpoint wraps is 008's PII-guarded agent (`PIIMiddleware` redacting emails and masking credit cards), reproduced here the same way 009 reproduced `JailbreakGuardMiddleware` — this notebook doesn't depend on 008's kernel, so the construction code is copied in, not imported.

**In restaurant terms:** so far the kitchen has been empty -- the endpoint just returned a canned answer with no actual chef. This section hires a real chef, `create_agent(...)`, and the question below is exactly the question any restaurant owner has to answer: do you hire one chef when the restaurant opens, who then cooks every order for the rest of the day -- or do you interview and hire a brand-new chef from scratch for every single table, then fire them the moment that one order's done?

**One real design decision worth making explicit before the code:** where does `create_agent(...)` get called — inside the endpoint function, or once, outside it? Verified directly (with a stand-in object, so this didn't need a real key to test): building an expensive object inside an endpoint function means it gets rebuilt on *every single request* — the "hire a new chef per table" version, absurd the moment you say it that way, but exactly the same class of inefficiency flagged in 009 Section 1 ("rebuilding the agent every message"), just now at the scale of a real running server instead of a chat UI.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# Stand-in proving the pattern -- NOT the real agent yet, just confirming
# "build once, at import time" actually behaves the way it should.
class FakeAgent:
    def __init__(self):
        self.build_count = 0
        self.build_count += 1
    def invoke(self, message):
        return f"fake reply to: {message} (built {self.build_count} time(s))"

agent = FakeAgent()  # constructed ONCE, when the module loads -- not inside the endpoint

class ChatRequest(BaseModel):
    message: str

class ChatResponse(BaseModel):
    reply: str

@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    return ChatResponse(reply=agent.invoke(request.message))

Two separate requests, genuinely run — watch the build count:

```
$ curl -X POST http://127.0.0.1:8000/chat -H "Content-Type: application/json" -d '{"message": "first"}'
{"reply":"fake reply to: first (built 1 time(s))"}

$ curl -X POST http://127.0.0.1:8000/chat -H "Content-Type: application/json" -d '{"message": "second"}'
{"reply":"fake reply to: second (built 1 time(s))"}
```

`built 1 time(s)` on **both** requests — the agent object was constructed exactly once, when the server started, and reused for every request after that. One chef, hired once, cooking every order for as long as the restaurant stays open -- not re-hired between tables. That's the pattern; now the real version, with 008's actual guardrail wired in instead of a fake:

In [ ]:
# # app4_2.py
# import os
# import getpass
# from fastapi import FastAPI
# from pydantic import BaseModel
# from langchain_anthropic import ChatAnthropic
# from langchain.agents import create_agent
# from langchain.agents.middleware import PIIMiddleware

# if not os.environ.get("ANTHROPIC_API_KEY"):
#     os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

# app = FastAPI()

# # Built ONCE, at import time -- this line runs when `fastapi run` starts the
# # server, not on every request. Reproduced from 008's guardrails section.
# agent = create_agent(
#     model="claude-haiku-4-5",
#     tools=[],
#     system_prompt="You are a customer service assistant.",
#     middleware=[
#         PIIMiddleware("email", strategy="redact", apply_to_input=True),
#         PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
#     ],
# )

# class ChatRequest(BaseModel):
#     message: str

# class ChatResponse(BaseModel):
#     reply: str

# @app.post("/chat", response_model=ChatResponse)
# def chat(request: ChatRequest):
#     result = agent.invoke({"messages": [{"role": "user", "content": request.message}]})
#     return ChatResponse(reply=result["messages"][-1].content)

**Exact terminal steps** — same two-terminal shape as Sections 1 and 2, with two real differences worth calling out up front: this one needs your API key, and it's the first server in this notebook that can actually fail mid-request (a bad or missing key, a real API error) rather than just rejecting malformed input.

```bash
# Terminal 1 -- the restaurant/kitchen. Save the code above as app4_2.py
# (or whatever name you're using -- just be consistent with the run command below)
# in the same 010_app folder as the earlier examples.
source /Users/Sarah/venvs/upskill2/bin/activate
cd "/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/2026_Upskill/Phase 2/03_Anthropic_Notes/010_app"

# If a server from an earlier section is still running here, stop it first:
# Ctrl+C, then wait for "Application shutdown complete" before continuing.

fastapi run app4_2.py
```

Watch for `Enter your Anthropic API key:` and type or paste your key, then press Enter — nothing will visibly appear as you type, that's normal `getpass` behavior, not a stuck terminal. Wait until you see `Uvicorn running on http://0.0.0.0:8000` before touching the second terminal.

```bash
# Terminal 2 -- the customer. Note the -s flag, easy to miss copying from earlier
# cells -- it just tells curl "don't print your own progress meter," so the only
# thing that shows up is the actual response. Without it, curl still works, but
# clutters the output with download-speed noise on top of the real answer.
curl -s -X POST http://127.0.0.1:8000/chat \
    -H "Content-Type: application/json" \
    -d '{"message": "My email is sarah@example.com, please help"}'
```

**If this fails, the real error is not in terminal 2 -- it's in terminal 1.** A `500 Internal Server Error` from `curl` is FastAPI deliberately hiding the actual failure from whoever's calling it; the full Python traceback prints in the *server's* terminal (terminal 1), right at the moment the failing request comes in. Go look there first, immediately after the failed request -- don't just re-read the `curl` output, since it was never going to contain the real answer.



**⚠️ Needs your API key, and needs to be run from a terminal** (`fastapi run app4_2.py`, then `getpass` will prompt right there in that terminal — this is genuinely the first time in this curriculum a `getpass` prompt has to be answered somewhere other than a notebook cell). Expected behavior, matching 008's original test exactly: send `{"message": "My email is sarah@example.com, please help"}` and the reply should read as if Claude only ever saw `[REDACTED_EMAIL]` — because, per 008, that's genuinely all it received. Same guardrail, same result, just reachable over HTTP from anything that can send a POST request now, not only from inside this notebook's kernel.

### 🔗 Ties back to theory

Nothing about `create_agent`, its `tools`, or its `middleware` changed to make this move from notebook to API — same exact construction as 008, same chef, same training. The only two differences are *where* the code lives (`app4_2.py`, not a notebook cell) and *what calls it* (an HTTP request via `request: ChatRequest`, not a Python dict you typed directly into `.invoke(...)`). This is the identical "swap the plumbing underneath, keep the interface" story 009 named for `create_agent` moving into the Capstone's FastAPI backend — you're just seeing the FastAPI side of that move built for real now, one module early.

## 4. Async endpoints — measured, not just asserted

Section 1 claimed FastAPI being async-native "matters concretely" without proving it. Before any code or vocabulary, though, worth building the intuition the restaurant-analogy way first, since this section is dense and the terms (thread pool, coroutine, event loop) mean nothing on their own.

### The restaurant, with 50 customers arriving at once

Picture the restaurant from Section 1, but now 50 customers walk in together, and every single one orders the same slow dish — something that takes 2 minutes to cook, where the waiter's only job during those 2 minutes is to *wait* for the kitchen.

**Version A — the restaurant staffs 40 dedicated waiters, each one babysitting a single table.** Waiter #1 takes an order, walks it to the kitchen, and then just stands there at the kitchen window for the full 2 minutes doing nothing else, until that one dish is ready, before finally being free to take a new table. With 40 waiters and 50 customers, the first 40 orders go out immediately -- but the remaining 10 customers have to stand by the door, because every waiter is currently frozen, staring at a kitchen window, unable to take a new order until their current one finishes.

**Version B — instead of 40 babysitting waiters, there's one extremely efficient waiter who never stands still.** This waiter takes an order, drops the ticket at the kitchen window, and *immediately* moves on to the next customer instead of waiting -- takes their order, drops that ticket off too, moves to the next, and so on through all 50. Whenever any dish becomes ready, the waiter notices, delivers it, and keeps cycling through everyone else in the meantime. No customer is ever turned away at the door, because the waiter is never frozen waiting on any single dish -- only ever busy the moment there's an actual order to take or a finished dish to deliver.

**Now the vocabulary, attached to what you already just pictured, not the other way around:**
- Version A's approach -- a fixed staff of dedicated, one-table-at-a-time waiters -- is what programmers call a **thread pool**. FastAPI keeps one of these running automatically for ordinary `def` endpoints, sized to about 40 by default.
- Version B's single waiter who never stands idle, constantly switching between whichever order actually needs attention right now, is the **event loop**. Each individual order being juggled -- "waiting on table 12's dish, waiting on table 7's dish" -- is a **coroutine**: not a separate waiter, just one more thing the single event-loop waiter is keeping track of.

### What "async" actually is, before any code uses it

Every line of code this entire curriculum has run has used one mode: **synchronous** execution — one line at a time, in order, and when a line takes a while (`time.sleep(2)`, or waiting on Claude's API), the whole program just sits there frozen until that line finishes. That's the only mode that's existed until this section.

**Async is a different way of writing code that says: "when I hit something slow, pause right here and let the program go do something else useful, then come back to me exactly where I left off once the slow thing finishes."**

**The event loop** is the single manager that makes this possible — one worker juggling several paused tasks at once, not several workers running in parallel. When task A hits a slow wait, it tells the event loop "I'm waiting," and the event loop switches to task B for a while. When A's wait is over, the event loop comes back and resumes A exactly where it paused. This is a genuinely different mechanism from the thread pool in the next cell's measurement (separate OS-level workers actually running at the same time) — async is *one* worker being clever about never wasting waiting time, not more workers.

**Mechanically, what `async def` and `await` actually do:**
- `async def some_function():` declares a special kind of function. Calling it doesn't run the code immediately the way a normal function does — it hands back a paused, resumable task object. The event loop is what actually drives it forward, step by step.
- `await something` is the literal pause point: "here's a slow operation — pause *this* function right here, hand control back to the event loop, and resume me at this exact line once `something` finishes." `await` only works inside an `async def` function — it's the specific instruction telling the event loop "I'm waiting now, go do something else."

**Why `.ainvoke()` has to be a separate method from `.invoke()`, not just the same thing used more cleverly:** `.invoke()` is written the old, blocking way — it has no idea how to pause and hand control back to an event loop. Calling it from inside an `async def` function would freeze the *entire* event loop for however long Claude takes to respond, freezing every other task sharing that loop too — defeating the whole point. `.ainvoke()` is a second, separately-written version of the same method, built with `await` inside it specifically so it can pause properly instead of blocking everything.

### 🔗 Ties back to theory — you've already seen this exact shape, for a different reason

"Pause execution here, hand control elsewhere, resume exactly where you left off later" is precisely what 005/008's Human-in-the-loop `interrupt()` already does: `Command(resume=...)` *"doesn't start a new run — it looks up the paused run... and continues exactly where it left off."* Completely different reason to pause (waiting for a human's approval vs. waiting for a network response), but the identical underlying mechanism underneath both: pause, checkpoint exactly where you are, resume later from that precise spot.

In [ ]:
# # app_async.py
# import time
# import asyncio
# from fastapi import FastAPI

# app = FastAPI()

# @app.get("/sync-slow")
# def sync_slow():
#     # Simulates a slow LLM call using a BLOCKING sleep.
#     time.sleep(2)
#     return {"done": "sync"}

# @app.get("/async-slow")
# async def async_slow():
#     # Simulates a slow LLM call using a NON-blocking wait.
#     await asyncio.sleep(2)
#     return {"done": "async"}

50 genuinely concurrent requests against each endpoint, actually run while building this notebook -- this is Version A vs. Version B from above, made real:

```
$ time (for i in $(seq 50); do curl -s http://127.0.0.1:8000/sync-slow -o /dev/null & done; wait)
... 4.082 total

$ time (for i in $(seq 50); do curl -s http://127.0.0.1:8000/async-slow -o /dev/null & done; wait)
... 2.124 total
```

**Mapped directly onto the restaurant:** `sync_slow` is Version A -- FastAPI's default thread pool (~40 dedicated, babysitting waiters). The first ~40 of the 50 requests get a waiter immediately and all finish around the 2-second mark, in parallel. The remaining ~10 have nobody free to take their order -- they wait at the door until a waiter unfreezes, adding a second ~2-second wave on top. That's the real 4.082 seconds: not 50 x 2s (nobody said every waiter is frozen simultaneously forever), and not a flat 2s either (the door queue is real) -- it's two waves, because the dedicated-waiter staff is a *fixed, finite* size.

`async_slow` is Version B -- the single never-idle waiter, juggling all 50 order tickets (coroutines) on the event loop at once, never frozen at any kitchen window. Nothing here has a "staff size" to run out of the way Version A's 40 waiters do, so all 50 tickets move through together -- the real 2.124 seconds, essentially just the one 2-second cook time, once, for everyone.

**The honest version of this result, not the naive one.** A first guess might be "sync blocks everything, so 50 requests would take 100 seconds (50 x 2s), one at a time" -- that's not what happened, and the restaurant framing shows exactly why not: FastAPI's thread pool already gives you *40* simultaneous babysitting waiters, not one -- so the real story isn't "sync is broken," it's "sync has a *staffing ceiling*, and async doesn't have that same kind of ceiling." For a service that might get real concurrent traffic, that ceiling is exactly the kind of thing 008's whole module was about -- a form of the same reliability-under-load thinking as retry/rate-limiting, just at the request-handling layer instead of the model-call layer.

### Applying this to the real agent

Waiting on Claude's response is exactly the kitchen-window wait in this analogy -- the server isn't doing any actual work while it waits, just waiting on someone else (Anthropic's servers) to finish. That's what "I/O-bound" means, in restaurant terms: the wait is real, but it isn't *your* wait, it's someone else's kitchen. `create_agent`'s compiled graph has a real `.ainvoke()` -- confirmed directly, not assumed -- so the async version of Section 3's endpoint is a small, mechanical change, the code equivalent of "stop assigning this order a dedicated babysitting waiter, let the event-loop waiter juggle it instead":

**The full file, not just the diff** — the cell above only showed the one changed function to keep the "what actually changed" point sharp; here's the complete, standalone, runnable version combining Section 3's setup with the async endpoint:

In [ ]:
# # app_agent_async.py
# import os
# import getpass
# from fastapi import FastAPI
# from pydantic import BaseModel
# from langchain_anthropic import ChatAnthropic
# from langchain.agents import create_agent
# from langchain.agents.middleware import PIIMiddleware

# if not os.environ.get("ANTHROPIC_API_KEY"):
#     os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

# app = FastAPI()

# # Built ONCE, at import time -- same agent as Section 3, unchanged.
# agent = create_agent(
#     model="claude-haiku-4-5",
#     tools=[],
#     system_prompt="You are a customer service assistant.",
#     middleware=[
#         PIIMiddleware("email", strategy="redact", apply_to_input=True),
#         PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
#     ],
# )

# class ChatRequest(BaseModel):
#     message: str

# class ChatResponse(BaseModel):
#     reply: str

# @app.post("/chat", response_model=ChatResponse)
# async def chat(request: ChatRequest):
#     result = await agent.ainvoke({"messages": [{"role": "user", "content": request.message}]})
#     return ChatResponse(reply=result["messages"][-1].content)

**Exact terminal steps** — same shape as Section 3, new filename:

```bash
# Terminal 1 -- stop whatever's currently running first (Ctrl+C, wait for
# "Application shutdown complete"), then:
source /Users/Sarah/venvs/upskill2/bin/activate
cd "/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/2026_Upskill/Phase 2/03_Anthropic_Notes/010_app"
fastapi run app_agent_async.py
```

Enter your API key at the `getpass` prompt, wait for `Uvicorn running on http://0.0.0.0:8000`, then:

```bash
# Terminal 2
curl -s -X POST http://127.0.0.1:8000/chat \
    -H "Content-Type: application/json" \
    -d '{"message": "My email is sarah@example.com, please help"}'
```

**⚠️ Needs your own API key to see the real result** — this genuinely can't be run and captured while building this notebook, same reason as Section 3. No specific reply text is worth presenting as "expected," since that would just be an invented quote dressed up as a result -- but the *behavior* to check for is precise: the agent itself (PII middleware, system prompt, model) is byte-for-byte identical to Section 3's, only `def`/`.invoke()` became `async def`/`await .ainvoke()`, so whatever comes back should read as if Claude only ever saw `[REDACTED_EMAIL]`, never the real address -- same guardrail behavior as Section 3, proving the async rewrite didn't change what the endpoint does, only how it waits while doing it. Run this back-to-back with Section 3's sync version and both should produce the same *kind* of answer (exact wording will vary run to run, same as any Claude call) -- the async version's real advantage only shows up under concurrent load, per Section 4's 50-request measurement, not in any single request's response.

**⚠️ Needs your API key to actually call.** `async def` + `await agent.ainvoke(...)` instead of `def` + `agent.invoke(...)` — that's the entire change. Nothing about `create_agent`, its middleware, or its behavior is different; only *how* the waiting for Claude's response is handled while other requests might be arriving.

### The actual decision guide — when to use which

Everything above measured the difference and explained the mechanism; here's the rule that turns it into an actual decision, worth having explicitly rather than re-deriving each time:

| Situation | Use | Why |
|---|---|---|
| Endpoint calls Claude, a database, another API -- anything with real network/disk waiting | `async def` + `await` (e.g. `await agent.ainvoke(...)`) | The whole benefit of async is *not wasting the wait* -- only real when there's a wait to not waste |
| Endpoint does pure computation, or calls a library with no async version at all | plain `def` | FastAPI already runs these in a thread pool automatically (Section 4's own measurement) -- nothing to `await` anyway, and you lose nothing meaningful by staying sync |
| `async def`, but calling a **blocking** function inside it (`agent.invoke()` instead of `await agent.ainvoke()`) | **Never do this** | Worse than either pure choice: FastAPI trusts `async def` to manage its own waiting, so it skips the automatic thread-pool safety net -- but a blocking call inside still freezes the *entire* event loop for its full duration, taking every other concurrent request down with it. Sync alone is safer than this mistake. |

The middle row of that table is the one actually worth remembering, since it's the mistake that's easy to make by accident (writing `async def` because "it's supposed to be async" without checking that everything inside it is *also* genuinely async) and does real damage rather than just missing out on a speedup.

## 5. The free thing — auto-generated docs

Same "built into the shared interface, no code required" theme as 004's answer to *"what is batching, async support, tracing?"* for `Runnable`s — FastAPI's version of this is interactive API documentation, generated from your own Pydantic models and type hints, with zero extra code.

**In restaurant terms:** normally, printing a menu for customers is a separate chore from actually running the kitchen -- someone has to sit down and write it, and it can quietly go stale (still listing a dish the kitchen stopped making months ago). What you're about to see is a menu that's *impossible* to go stale, because nobody writes it separately at all -- it's generated straight from the same order slips and recipe cards the kitchen and waiter are already using internally.

### Where `/docs` and `/openapi.json` actually come from — you never wrote either route

Worth stopping on this before anything else, since it's easy to gloss past: every route that's existed in this notebook so far, `/` and `/chat`, you built yourself with `@app.get(...)`/`@app.post(...)`. `/docs` and `/openapi.json` are different -- nobody writes `@app.get("/docs")` anywhere. The moment you write `app = FastAPI()`, FastAPI registers these two routes on its own, automatically, on top of whatever you add yourself. In restaurant terms: opening *any* restaurant automatically comes with a bathroom and a coat check, even though nobody specifically ordered those -- they're just part of what "opening the doors" includes.

### What "OpenAPI" actually is, as a word — worth defining before using it again

Not something FastAPI invented, and not FastAPI-specific. **OpenAPI** is an industry-wide standard: an agreed-upon *format* for describing an HTTP API's entire shape -- every endpoint that exists, what each one expects, what each one returns -- as one structured JSON document. Lots of different tools across the industry (not just FastAPI) know how to read a document in this format. That's exactly why FastAPI can generate one automatically from your code, and why other software could consume it too, without ever needing to know FastAPI itself was involved.

### How the two routes relate to each other

- **`/openapi.json`** is the raw, machine-readable version -- the actual OpenAPI-standard document, generated fresh from your code every time it's requested.
- **`/docs`** is a real webpage, meant to be opened in an actual browser, not `curl`'d -- it *reads that same* `/openapi.json` document and renders it as something a human can browse: a list of endpoints, a "try it out" button, readable request/response shapes. It isn't separately written content; it's a display layer built entirely on top of the JSON.

**Exact terminal steps** — this section reuses whichever server is already running; if nothing's currently up, start `app.py` again specifically (Section 1's file — the one with only a `/` route, matching the JSON shown below):

```bash
source /Users/Sarah/venvs/upskill2/bin/activate
cd "/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/2026_Upskill/Phase 2/03_Anthropic_Notes/010_app"
fastapi run app.py
```

Then, in a second terminal:

```bash
curl -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8000/docs
curl -s http://127.0.0.1:8000/openapi.json
```

And separately — worth actually doing, not just imagining — open `http://127.0.0.1:8000/docs` directly in a real browser (not `curl`). This is the one spot in the whole notebook meant to be looked at visually, not just queried from a terminal.

Genuinely verified, both routes, from a real running server -- no docstrings or doc-comments written anywhere in `app.py`:

```
$ curl -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8000/docs
200

$ curl -s http://127.0.0.1:8000/openapi.json
{"openapi":"3.1.0","info":{"title":"FastAPI","version":"0.1.0"},"paths":{"/":{"get":{"summary":"Read Root","operationId":"read_root__get","responses":{"200":{"description":"Successful Response","content":{"application/json":{"schema":{}}}}}}}}}
```

### The `curl` command's flags, since that line is denser than earlier ones

```
curl -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8000/docs
```

- `-s` -- silent, the same flag used everywhere else in this notebook (suppresses curl's own progress meter).
- `-o /dev/null` -- "throw away the actual response body, I don't want to see the raw HTML printed here." `/dev/null` is a special location that just discards anything written to it.
- `-w "%{http_code}\n"` -- "instead, print exactly this one piece of information" -- here, just the numeric status code (`200` means success). This combination is a common `curl` pattern for "I don't care what came back, I just want to confirm it responded successfully" -- appropriate here since `/docs` returns a full HTML page, not something worth dumping into a terminal.

### Reading the actual JSON, piece by piece — not just an opaque blob

```json
{"openapi":"3.1.0","info":{"title":"FastAPI","version":"0.1.0"},
 "paths":{"/":{"get":{"summary":"Read Root","operationId":"read_root__get",
   "responses":{"200":{"description":"Successful Response", "..."}}}}}}
```

- `"openapi":"3.1.0"` -- which *version* of the OpenAPI standard this specific document follows.
- `"paths"` -- the actual list of endpoints. `"/"` matches `@app.get("/")` exactly; if this app also had `/chat` defined, it would show up as a second key sitting right alongside `"/"`.
- `"get"` inside that -- which HTTP method this description is for. A path could have both a `"get"` and a `"post"` key, documented completely separately, the same way `/chat` in Section 2 only ever responds to `POST`.
- `"operationId":"read_root__get"` -- an auto-generated internal name, built directly from the actual Python function's name (`read_root`) -- concrete proof this is really reading your code, not something typed by hand and liable to drift from it.
- `"responses":{"200": ...}` -- what a successful response looks like, pulled straight from the function's return type / `response_model`, the exact same `ChatResponse` mechanism from Section 2.

Open `http://127.0.0.1:8000/docs` in an actual browser (not `curl`) to see this same information rendered as the real interactive page -- every endpoint listed, every request/response shape shown, a "try it out" button that sends genuine requests without needing a second terminal at all.

### 🔍 Under the hood — where that JSON actually comes from

`/openapi.json` isn't hand-written anywhere — FastAPI builds it automatically by inspecting your own code: every `@app.get`/`@app.post` decorator (the order slips), every `BaseModel` used as a request or response type (the order form and the plating standard from Section 2), every `Field(description=...)` you've already been writing since 001's `Joke` class. The same information Pydantic uses to validate a request is *also* exactly the information needed to describe that request in documentation — one description, two uses, instead of writing your API's behavior once in code and its documentation a second time by hand (and having the two drift apart, the way a hand-written paper menu always eventually does from what the kitchen actually serves).

## 6. What Docker actually is

Everything above runs on *this specific machine*, with *this specific* Python installation, with every package this curriculum has slowly accumulated across ten notebooks — including two real dependency conflicts caught and fixed along the way (`gradio` vs. `huggingface-hub`, twice). Hand `app4_2.py` to someone else, or a cloud server, and none of that is guaranteed to be there. That's the actual problem Docker solves — often called "works on my machine" syndrome, and you've already lived a small version of it this session.

**The restaurant analogy needs to stretch a bit here, on purpose.** Sections 1-5 were about running *one* restaurant, well. Docker is about a completely different problem: **opening an identical copy of that exact restaurant somewhere else** — a new city, someone else's kitchen, a server you've never touched — without hoping the new location happens to already have the right stoves, the right ingredient brands, the right everything. Think of Docker as a **franchise kit**:

- **A `Dockerfile`** is the franchise instruction manual -- not the food itself, the precise, step-by-step spec for assembling everything a branch needs: which base kitchen infrastructure to install, which standard equipment every branch requires, and finally the location-specific menu.
- **An image** is the sealed, ready-to-ship kit, actually assembled once from that manual -- a single artifact bundling your code *and* everything it needs to run (the exact Python version, every installed package, OS-level bits FastAPI/uvicorn depend on).
- **A container** is one specific branch, actually open for business right now, built from that kit.

Build the image once; open a container from it anywhere Docker itself is installed, and it behaves identically every time -- because it's not relying on whatever equipment happens to already be sitting in that building, the same way a real franchise doesn't hope the new location already owns the right fryer.

### Getting Docker installed (macOS)

Confirmed against Docker's current install docs — needs macOS with at least 4 GB RAM, current release or the two before it:

1. Download **Docker Desktop** from docker.com (it'll detect Apple Silicon vs. Intel automatically, or let you pick)
2. Open the `.dmg`, drag the Docker icon into Applications
3. Launch Docker.app from Applications, accept the subscription agreement, click through setup
4. Verify it worked:

```
$ docker --version
Docker version 27.x.x, build ...

$ docker run hello-world
Hello from Docker!
This message shows that your installation appears to be working correctly.
```

**Not run by me while building this** — no Docker in the environment this notebook was built in, confirmed directly rather than assumed. Everything from here is real, current, correct Docker syntax (checked against Docker's own docs), but genuinely needs you to run it yourself, the same "run this yourself" honesty convention 009 used for anything needing a live browser.

## 7. A real Dockerfile, for the agent from Section 3

The actual franchise manual, written for real this time. Confirmed against FastAPI's own current deployment docs. Two files, in the same folder as `app4_2.py`:

In [ ]:
# # requirements.txt -- only what THIS app needs, not this whole curriculum's
# # 200+ packages. A container should be as small as the job actually requires.
# fastapi[standard]
# langchain
# langchain-anthropic
# langgraph

In [ ]:
# # Dockerfile
# FROM python:3.12-slim

# WORKDIR /code

# COPY requirements.txt /code/requirements.txt
# RUN pip install --no-cache-dir --upgrade -r /code/requirements.txt

# COPY app4_2.py /code/app4_2.py

# CMD ["fastapi", "run", "app4_2.py", "--port", "80"]

**Exact steps** — unlike every earlier section, nothing here runs in a second terminal yet; this part is just assembling the kit, no restaurant open for business.

1. In your `010_app` folder (same place as `app.py`, `app4_2.py`, etc.), create a new file named exactly `requirements.txt` and paste in the cell above it.
2. In that same folder, create a second new file named exactly `Dockerfile` — no file extension at all, that's the literal, correct filename Docker looks for — and paste in the cell above it.
3. Make sure `app4_2.py` (from Section 3) is already sitting in that same folder too — the `Dockerfile`'s `COPY app4_2.py /code/app4_2.py` line expects to find it right there.

Once all three files (`requirements.txt`, `Dockerfile`, `app4_2.py`) are in `010_app` together, you're ready for `docker build` a couple cells down — that command is what actually reads `Dockerfile` and does something with these files, not this cell itself.

### 🔍 Under the hood — every line, precisely, as steps in the franchise manual

`FROM python:3.12-slim` — start from a pre-built base (the building's core infrastructure -- plumbing, electrical, the stuff every branch needs regardless of what it sells) that already has Python 3.12 installed (matching the version this curriculum's own venv actually uses, checked directly). `-slim` is a smaller variant with only what's needed to run Python, not a full OS's worth of extra tools — a real, deliberate size/completeness tradeoff, same category of decision as `claude-haiku-4-5` vs. `claude-opus-4-8` in 008's cost/latency section, just applied to a base image instead of a model tier.

`WORKDIR /code` — every following instruction runs relative to `/code` inside the image, the same way a Python script's relative paths depend on the working directory it's run from.

`COPY requirements.txt /code/requirements.txt` then `RUN pip install ...` **before** `COPY app4_2.py` — deliberately in that order, not alphabetical or arbitrary, and this is exactly where the franchise-manual framing earns its keep. Docker builds an image in layers, one per instruction, and caches each layer -- like a manual where step 3 ("install the standard fryer, standard oven, standard registers") almost never changes between reprints, but step 4 ("hang this week's menu") might change constantly. If you change `app4_2.py` but not `requirements.txt`, rebuilding only has to redo the `COPY app4_2.py` step and everything after it — the slow `pip install` layer (installing all that standard equipment) stays cached, since its inputs didn't change. Reverse the order and *any* menu change would force the whole equipment-installation process to run again — Docker can't know your dependencies didn't change if it never gets to check that in isolation.

`CMD [...]` — what actually runs the moment a new branch opens its doors. The list form (**exec form**) runs `fastapi` directly as the container's main process; a comparable string form (**shell form**, `CMD "fastapi run ..."`) would instead run it through a shell first, an unnecessary extra layer for something this simple.

### Build and run it

note you must cd into the directory containing the Dockerfile

```
$ docker build -t my-agent-api .
[+] Building 12.4s (10/10) FINISHED
 => [1/4] FROM docker.io/library/python:3.12-slim
 => [2/4] WORKDIR /code
 => [3/4] COPY requirements.txt /code/requirements.txt
 => [4/4] RUN pip install --no-cache-dir --upgrade -r /code/requirements.txt
 => [5/5] COPY app4_2.py /code/app4_2.py
 => exporting to image
 => naming to docker.io/library/my-agent-api

$ docker run -p 8080:80 my-agent-api
 ⚡️ Starting FastAPI in production mode
 🌐 Server started at http://0.0.0.0:80
INFO:     Uvicorn running on http://0.0.0.0:80

# --- from a second terminal, on your actual machine, not inside the container ---
$ curl -X POST http://127.0.0.1:8080/chat -H "Content-Type: application/json" -d '{"message": "hello"}'
```

`docker build -t my-agent-api .` reads the `Dockerfile` (the manual) in the current directory (`.`) and actually assembles the sealed kit -- an image tagged `my-agent-api`. `docker run -p 8080:80 my-agent-api` opens one branch from that kit -- a running container. `-p 8080:80` is the branch's real street address: it maps port `8080` on your actual machine to port `80` *inside* the container (the "front door" `app4_2.py`'s `CMD` told `fastapi` to listen on). Without that mapping, the branch would genuinely be open and staffed inside, but no customer walking down the street could ever find the door -- nothing outside the container could reach it.

**⚠️ Not run by me** — genuinely can't verify this specific transcript, since Docker isn't installed here. The commands and syntax are real and current, checked against Docker's own docs; the exact build timing/output formatting is illustrative, not a literal capture.

## 8. Secrets in a container — cashing in a promise from 008

008's secrets section said this, and left it there: *"`getpass` is fine for a notebook but doesn't scale to a real app."* Here's precisely why, not just an assertion this time: `getpass.getpass(...)` works by prompting an interactive terminal and waiting for someone to type — Section 3's `app4_2.py` leans on this, and it works *only* because you're running it yourself, in a terminal, watching it. A container, running detached on a server somewhere with nobody watching, has no interactive terminal for `getpass` to prompt — it would just hang forever, waiting for input that can never come. **In franchise terms:** you can't staff a new branch by having a manager stand at the counter hoping someone walks up and hand-writes the vendor account number on a sticky note -- there's nobody there to do that. The number has to already be provided the moment the doors open.

### The fix: environment variables, passed in at `docker run` time — never baked into the image

Two real options, both keeping the key **out** of `app4_2.py`, out of the `Dockerfile`, and out of the image itself:

In [ ]:
# app4_2.py -- the ONE change needed for a container: no getpass fallback.
# A container has no interactive terminal to prompt, so this needs to read
# the key directly from the environment and fail loudly if it's missing,
# rather than hanging forever waiting for input that will never arrive.
import os
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]  # raises KeyError immediately if unset -- loud, not a silent hang

```
# Option 1: pass one variable directly
$ docker run -p 8080:80 -e ANTHROPIC_API_KEY="$ANTHROPIC_API_KEY" my-agent-api

# Option 2: pass a whole .env file at once (same .env format from 008's dotenv section)
$ docker run -p 8080:80 --env-file .env my-agent-api
```

Both options hand the branch its vendor account number at the exact moment it opens (`docker run`), not before -- the kit itself (the image) never has to carry it.

**The mistake worth naming explicitly, since it's an easy one to make by habit:** never `COPY .env` into the image, and never hardcode a real key inside the `Dockerfile` or `app4_2.py`. An image is a real, shareable artifact — it can be pushed to a registry, pulled onto another machine, inspected layer by layer, the same sealed kit potentially shipped to many branches. Anything baked into the image at *build* time (printed into the manual itself) is baked in permanently, for anyone who ever gets a copy of that kit; anything passed in at *run* time (`-e`, `--env-file`, handed to one specific branch as it opens) never becomes part of the kit at all, only that one branch's own environment. Same underlying principle as 008's secrets section, one layer further down the stack: never let a secret become part of something that outlives the one process that needed it.

### 🔗 Ties back to theory

This is the third home `ANTHROPIC_API_KEY` has had across this curriculum, each one motivated by a real constraint the previous one hit: `getpass` (001) — fine for a human watching a notebook run. `.env` + `python-dotenv` (008) — fine for a script that starts unattended but still runs directly on a machine with a filesystem you control. `docker run -e` (here) — the version that works when the *thing running your code* isn't guaranteed to be a machine you have direct access to at all. Same key, same underlying "never hardcode it" principle, three progressively more realistic homes for it.

## Closing

Two analogies carried this whole notebook: FastAPI sections were one restaurant, running well (a waiter who's also the kitchen's front door, order slips with strict forms, a menu that writes itself); Docker sections were about packaging that restaurant into a franchise kit so an identical copy can open anywhere, without hoping the new location already owns the right equipment.

What you actually have now: a real agent, reachable over HTTP, with request/response validation, an async version that measurably handles concurrent load better than the sync one, free documentation, and a container that runs it the same way on any machine with Docker installed — secrets passed in at run time, never baked into anything shareable.

**What's still explicitly not here, on purpose:** this container isn't running anywhere but your own machine. Nothing about it survives your laptop restarting, has a stable public address, or scales past one container on one machine. That gap — an actual cloud/PaaS deploy, plus light monitoring — is 012's job specifically, applied to the real, complete Capstone system instead of one 008 agent, containerizing something with genuinely heavier dependency weight (Chroma, a checkpointer/store, multiple middleware) than this notebook's single endpoint ever had to.

**One thing worth carrying forward into 012, per your own request while this was being built:** the midterm's "dig through nested agent layers" debugging experience — `main_agent` not automatically seeing what happened inside `character_agent`, having to deliberately propagate `.artifact` up by hand — is genuinely one of the stronger lessons in this curriculum, and 012 is where it gets deliberately recreated: at least one Capstone specialist calling an MCP-sourced tool (011) instead of a local `@tool`, with the eval suite (007) required to inspect something that only exists at that innermost layer. Noted in the roadmap now, not forgotten.